# NC-SSM Vision: Tunnel & Night Vision AI
**Target: ICCV 2027** | Ultra-Lightweight Edge Vision

- **Backbone**: NC-SSM Vision (Noise-Conditioned Selectivity-Modulated SSM)
- **Task 1**: Lane Detection (Row-Anchor Regression)
- **Task 2**: Critical Object Spotting (CenterNet-style)
- **Model Size**: 25K params / 24.4 KB INT8

---
**Runtime > Change runtime type > T4 GPU** before running.

## 1. Setup: Clone Repo & Install

In [ ]:
# Clone repository
!git clone https://github.com/DrJinHoChoi/NC-SSM-Vision-ICCV2027.git
%cd NC-SSM-Vision-ICCV2027
!git log --oneline -3

In [ ]:
# Install dependencies
!pip install -q torch torchvision torchaudio
import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 2. Verify NC-SSM Vision Backbone

In [ ]:
from ncssm_vision import NanoMambaVision

# Model verification
configs = {
    'Nano':  dict(d_model=16, d_state=4, n_repeats=4),
    'Tiny':  dict(d_model=24, d_state=6, n_repeats=4),
    'Small': dict(d_model=32, d_state=8, n_repeats=4),
    'Base':  dict(d_model=48, d_state=16, n_repeats=6),
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.rand(2, 3, 288, 288, device=device)

print(f"{'Model':<8} | {'Params':>8} | {'INT8 KB':>8} | {'Output':>15}")
print('-' * 50)
for name, cfg in configs.items():
    model = NanoMambaVision(
        img_size=288, patch_size=16, in_chans=3, n_classes=10,
        d_conv=3, expand=1.0, n_layers=1, weight_sharing=True,
        use_dual_norm=True, use_retinex=True, use_nc_ssm=True,
        **cfg
    ).to(device)
    model.eval()
    p = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        out = model(x)
    print(f"{name:<8} | {p:>8,} | {p/1024:>7.1f} | {str(list(out.shape)):>15}")
    del model

print('\nBackbone verified on', device)

## 3. Task Head Verification

In [ ]:
from ncssm_vision_tasks import (
    create_lane_detector_tiny, create_critical_detector_tiny,
    create_multitask_detector_tiny
)

x = torch.rand(2, 3, 288, 288, device=device)

models_info = [
    ('LaneDet-Tiny', create_lane_detector_tiny),
    ('CriticalDet-Tiny', create_critical_detector_tiny),
    ('MultiTask-Tiny', create_multitask_detector_tiny),
]

print(f"{'Model':<20} | {'Total':>8} | {'Backbone':>8} | {'Head':>6} | {'INT8':>7}")
print('-' * 65)
for name, fn in models_info:
    m = fn(img_size=288).to(device)
    m.eval()
    total = sum(p.numel() for p in m.parameters())
    bb = sum(p.numel() for p in m.backbone.parameters())
    with torch.no_grad():
        _ = m(x)
    print(f"{name:<20} | {total:>8,} | {bb:>8,} | {total-bb:>6,} | {total/1024:>5.1f}KB")
    del m

torch.cuda.empty_cache() if torch.cuda.is_available() else None

## 4. Train: Lane Detection (Synthetic)

In [ ]:
import time
import json
from torch.utils.data import DataLoader
from train_vision import (
    SyntheticLaneDataset, SyntheticDetectionDataset,
    train_one_epoch, evaluate, evaluate_conditions,
    EMA, LaneDetectionLoss, CriticalObjectLoss
)
from ncssm_vision_tasks import create_lane_detector_tiny

# --- Config ---
TASK = 'lane'  # 'lane' or 'detection'
EPOCHS = 30
BATCH_SIZE = 64
LR = 3e-3
IMG_SIZE = 288
N_TRAIN = 5000
N_VAL = 1000

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

# Dataset
train_ds = SyntheticLaneDataset(n_samples=N_TRAIN, img_size=IMG_SIZE, condition='mixed', split='train')
val_ds = SyntheticLaneDataset(n_samples=N_VAL, img_size=IMG_SIZE, condition='mixed', split='val')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Data: {len(train_ds)} train, {len(val_ds)} val')

# Model
model = create_lane_detector_tiny(img_size=IMG_SIZE).to(device)
criterion = LaneDetectionLoss(sim_weight=1.0)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {params:,} params ({params/1024:.1f} KB INT8)')

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
ema = EMA(model, decay=0.999)

# Training
history = []
best_acc = 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, epoch, task='lane')
    ema.update()
    ema.apply_shadow()
    val_loss, val_acc = evaluate(model, val_loader, criterion, device, task='lane')
    ema.restore()
    scheduler.step()
    dt = time.time() - t0
    lr = scheduler.get_last_lr()[0]
    
    marker = ''
    if val_acc > best_acc:
        best_acc = val_acc
        ema.apply_shadow()
        torch.save(model.state_dict(), 'lane_best.pt')
        ema.restore()
        marker = ' *BEST*'
    
    print(f'  E{epoch:>2}/{EPOCHS} | loss={train_loss:.4f} | val={val_loss:.4f} | '
          f'acc={val_acc:.1f}% | lr={lr:.1e} | {dt:.0f}s{marker}')
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc})

print(f'\nBest accuracy: {best_acc:.1f}%')

## 5. Per-Condition Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('lane_best.pt', map_location=device))
model.eval()

print('Per-Condition Evaluation (Lane Detection):')
print('=' * 55)
results = evaluate_conditions(model, IMG_SIZE, 18, device, task='lane')

print(f"\n{'Condition':<12} | {'Accuracy':>10} | Audio Analog")
print('-' * 55)
analogs = {
    'normal': 'Clean audio',
    'dark': 'Factory noise (stationary)',
    'glare': 'Impulse noise (non-stationary)',
    'fog': 'White noise (broadband)',
    'shadow': 'Narrowband noise (partial)',
}
for cond, res in results.items():
    print(f"  {cond:<10} | {res['accuracy']:>8.1f}% | {analogs[cond]}")

## 6. Train: Critical Object Detection (Synthetic)

In [ ]:
from ncssm_vision_tasks import create_critical_detector_tiny

# Detection config
DET_EPOCHS = 30

# Dataset
det_train_ds = SyntheticDetectionDataset(n_samples=N_TRAIN, img_size=IMG_SIZE, condition='mixed', split='train')
det_val_ds = SyntheticDetectionDataset(n_samples=N_VAL, img_size=IMG_SIZE, condition='normal', split='val')
det_train_loader = DataLoader(det_train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
det_val_loader = DataLoader(det_val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Model
det_model = create_critical_detector_tiny(img_size=IMG_SIZE).to(device)
det_criterion = CriticalObjectLoss()
det_params = sum(p.numel() for p in det_model.parameters())
print(f'Detection Model: {det_params:,} params ({det_params/1024:.1f} KB INT8)')

# Optimizer
det_optimizer = torch.optim.AdamW(det_model.parameters(), lr=LR, weight_decay=0.01)
det_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(det_optimizer, T_max=DET_EPOCHS, eta_min=1e-5)
det_ema = EMA(det_model, decay=0.999)

det_history = []
det_best_acc = 0

for epoch in range(1, DET_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(det_model, det_train_loader, det_criterion, det_optimizer, device, epoch, task='detection')
    det_ema.update()
    det_ema.apply_shadow()
    val_loss, val_acc = evaluate(det_model, det_val_loader, det_criterion, device, task='detection')
    det_ema.restore()
    det_scheduler.step()
    dt = time.time() - t0
    
    marker = ''
    if val_acc > det_best_acc:
        det_best_acc = val_acc
        det_ema.apply_shadow()
        torch.save(det_model.state_dict(), 'det_best.pt')
        det_ema.restore()
        marker = ' *BEST*'
    
    print(f'  E{epoch:>2}/{DET_EPOCHS} | loss={train_loss:.4f} | val={val_loss:.4f} | '
          f'recall={val_acc:.1f}% | {dt:.0f}s{marker}')
    det_history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss, 'val_acc': val_acc})

print(f'\nBest detection recall: {det_best_acc:.1f}%')

## 7. Detection Per-Condition Evaluation

In [ ]:
det_model.load_state_dict(torch.load('det_best.pt', map_location=device))
det_model.eval()

print('Per-Condition Evaluation (Critical Object Detection):')
print('=' * 55)
det_results = evaluate_conditions(det_model, IMG_SIZE, 18, device, task='detection')

print(f"\n{'Condition':<12} | {'Recall':>10} | Audio Analog")
print('-' * 55)
for cond, res in det_results.items():
    print(f"  {cond:<10} | {res['accuracy']:>8.1f}% | {analogs[cond]}")

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('NC-SSM Vision Training (Synthetic)', fontsize=14, fontweight='bold')

# Lane Loss
epochs_lane = [h['epoch'] for h in history]
axes[0,0].plot(epochs_lane, [h['train_loss'] for h in history], 'b-', label='Train')
axes[0,0].plot(epochs_lane, [h['val_loss'] for h in history], 'r--', label='Val')
axes[0,0].set_title('Lane Detection Loss')
axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Loss'); axes[0,0].legend(); axes[0,0].grid(alpha=0.3)

# Lane Accuracy
axes[0,1].plot(epochs_lane, [h['val_acc'] for h in history], 'g-o', markersize=3)
axes[0,1].set_title(f'Lane Detection Accuracy (best={best_acc:.1f}%)')
axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Accuracy (%)'); axes[0,1].grid(alpha=0.3)

# Detection Loss
epochs_det = [h['epoch'] for h in det_history]
axes[1,0].plot(epochs_det, [h['train_loss'] for h in det_history], 'b-', label='Train')
axes[1,0].plot(epochs_det, [h['val_loss'] for h in det_history], 'r--', label='Val')
axes[1,0].set_title('Object Detection Loss')
axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Loss'); axes[1,0].legend(); axes[1,0].grid(alpha=0.3)

# Detection Recall
axes[1,1].plot(epochs_det, [h['val_acc'] for h in det_history], 'm-o', markersize=3)
axes[1,1].set_title(f'Object Detection Recall (best={det_best_acc:.1f}%)')
axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Recall (%)'); axes[1,1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

## 9. Per-Condition Bar Chart

In [ ]:
import numpy as np

conditions = ['normal', 'dark', 'glare', 'fog', 'shadow']
lane_accs = [results[c]['accuracy'] for c in conditions]
det_accs = [det_results[c]['accuracy'] for c in conditions]

x = np.arange(len(conditions))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, lane_accs, width, label='Lane Detection', color='#0891B2')
bars2 = ax.bar(x + width/2, det_accs, width, label='Object Detection', color='#F59E0B')

ax.set_ylabel('Accuracy / Recall (%)')
ax.set_title('NC-SSM Vision: Per-Condition Performance\n(Analogous to Audio Noise Robustness Table)')
ax.set_xticks(x)
ax.set_xticklabels([f'{c}\n({analogs[c].split("(")[0].strip()})' for c in conditions], fontsize=9)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1 + bars2:
    h = bar.get_height()
    ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('per_condition_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: per_condition_results.png')

## 10. Model Profiling (Latency & MACs)

In [ ]:
import time

# Latency benchmark
model.eval()
x_bench = torch.rand(1, 3, 288, 288, device=device)

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(x_bench)
if device == 'cuda':
    torch.cuda.synchronize()

# Benchmark
n_runs = 100
t0 = time.time()
for _ in range(n_runs):
    with torch.no_grad():
        _ = model(x_bench)
if device == 'cuda':
    torch.cuda.synchronize()
latency_ms = (time.time() - t0) / n_runs * 1000

print(f'Inference Latency: {latency_ms:.2f} ms ({1000/latency_ms:.0f} FPS) on {device}')
print(f'Model: {sum(p.numel() for p in model.parameters()):,} params')
print(f'INT8: {sum(p.numel() for p in model.parameters())/1024:.1f} KB')

# Memory
if device == 'cuda':
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        _ = model(x_bench)
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    print(f'Peak GPU Memory: {peak_mb:.1f} MB')

## 11. Save Results to Google Drive (Optional)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

save_dir = '/content/drive/MyDrive/NC-SSM-Vision-Results'
os.makedirs(save_dir, exist_ok=True)

# Save models
for f in ['lane_best.pt', 'det_best.pt', 'training_curves.png', 'per_condition_results.png']:
    if os.path.exists(f):
        shutil.copy(f, save_dir)
        print(f'Saved: {f} -> {save_dir}')

# Save history
with open(f'{save_dir}/lane_history.json', 'w') as f:
    json.dump(history, f, indent=2)
with open(f'{save_dir}/det_history.json', 'w') as f:
    json.dump(det_history, f, indent=2)

# Save per-condition results
all_results = {
    'lane': {k: v for k, v in results.items()},
    'detection': {k: v for k, v in det_results.items()}
}
with open(f'{save_dir}/per_condition_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'\nAll results saved to: {save_dir}')